In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.decomposition import PCA

from sklearn.cluster import OPTICS
from sklearn.metrics import (
    silhouette_score, 
    calinski_harabasz_score, 
    davies_bouldin_score)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [17]:
class DataLoader:
    def __init__(self,data:str):
        self.data = pd.read_csv(data)

    def get_data(self):
        return self.data.head()

    def get_summary(self):
        return self.data.info()

class Preprocessing(DataLoader):
    def __init__(self,data:str):
        super().__init__(data)

    def delete_col(self,col:list):
        self.data = self.data.drop(columns=col)
        return self

    def transform_data(self):
        self.num_cols = self.data.select_dtypes(include=np.number).columns
        self.cat_cols = self.data.select_dtypes(include='object').columns

        num_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        cat_pipeline = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder",OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
        self.preprocessor = ColumnTransformer(transformers=[
            ('num',num_pipeline,self.num_cols),
            ('cat',cat_pipeline,self.cat_cols)
        ])

        self.preprocessor.set_output(transform='pandas')
        self.data = self.preprocessor.fit_transform(self.data)
        return self.data

In [18]:
processor = Preprocessing('00.dataset/Mall_Customers.csv')

remove_columns = processor.delete_col(col=["CustomerID"])

df_clean = remove_columns.transform_data()

df_clean.head()

,num__Age,num__Annual Income (k$),num__Spending Score (1-100),cat__Gender_Female,cat__Gender_Male
0,-1.424569,-1.738999,-0.434801,0.0,1.0
1,-1.281035,-1.738999,1.195704,0.0,1.0
2,-1.352802,-1.700830,-1.715913,1.0,0.0
3,-1.137502,-1.700830,1.040418,1.0,0.0
4,-0.563369,-1.662660,-0.395980,1.0,0.0


In [ ]:
class OPTICSGridSearch:
    def __init__(self,min_samples:int,xi:float,min_cluster_size:float):
        self.min_samples = min_samples
        self.xi = xi
        self.min_cluster_size = min_cluster_size

    def fit_evaluted(self,X,eps_range = np.arange(0.1, 1.6, 0.1)):
        X_arr = X.values if isinstance(X, pd.DataFrame) else X # Konversi ke numpy array jika input berupa DataFrame

        eps_values = []
        sil_scores = []
        ch_scores = []
        db_scores = []
        n_clusters_found = []

        for eps in eps_range:
            optics = OPTICS(min_samples=self.min_samples,xi=self.xi,min_cluster_size=self.min_cluster_size,max_eps=eps,)
            labels = optics.fit_predict(X_arr)

            # Pisahkan noise (-1) dari klaster valid
            core_mask = labels != -1
            unique_labels = set(labels[core_mask])

            if len(unique_labels) >= 2:
                X_valid = X_arr[core_mask]
                labels_valid = labels[core_mask]

                eps_values.append(round(eps, 2))
                sil_scores.append(silhouette_score(X_valid, labels_valid))
                ch_scores.append(calinski_harabasz_score(X_valid, labels_valid))
                db_scores.append(davies_bouldin_score(X_valid, labels_valid))
                n_clusters_found.append(len(unique_labels))

                self.df_scores = pd.DataFrame({
                'Clusters Found': n_clusters_found,
                'eps': eps_values,
                'Silhouette Score': sil_scores,
                'Calinski-Harabasz Score': ch_scores,
                'Davies-Bouldin Score': db_scores,
                })

            if not self.df_scores.empty():
                scaler = MinMaxScaler()
                norm_sil = scaler.fit_transform(self.df_scores[['Silhouette Score']])
                norm_ch = scaler.fit_transform(self.df_scores[['Calinski-Harabasz Score']])
                norm_db = 1 - scaler.fit_transform(self.df_scores[['Davies-Bouldin Score']])
                self.df_scores['Composite_Score'] = (norm_sil + norm_ch + norm_db) / 3

        return self

    def plot_evaluation(self,figsize=(15, 11)):
        
    